In [2]:
import pandas as pd
users = pd.read_csv(r'C:\Users\miml4\Desktop\Learning\Courses\Project\Project_03\data\users.csv', usecols=['user_id', 'state', 'current_plan'])


In [3]:
paid_users_count = users[users['current_plan'].notnull()].shape[0]
total_users_count = users.shape[0]
conversion_rate = (paid_users_count / total_users_count) * 100

print(f"전체 유저 수: {total_users_count:,}명")
print(f"유료 결제 유저 수: {paid_users_count:,}명")
print(f"유료 전환율(CVR): {conversion_rate: .2f}%")

전체 유저 수: 19,050명
유료 결제 유저 수: 5,527명
유료 전환율(CVR):  29.01%


In [4]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
users = pd.read_csv(r'C:\Users\miml4\Desktop\Learning\Courses\Project\Project_03\data\users.csv')
event_logs = pd.read_csv(r'C:\Users\miml4\Desktop\Learning\Courses\Project\Project_03\data\event_logs.csv')
chat_events = pd.read_csv(r'C:\Users\miml4\Desktop\Learning\Courses\Project\Project_03\data\chat_events.csv')
courses = pd.read_csv(r'C:\Users\miml4\Desktop\Learning\Courses\Project\Project_03\data\courses.csv')

In [6]:
import pandas as pd
import json
from datetime import timedelta

# [중요] 데이터프레임 변수명(users, event_logs 등)이 실제 환경과 동일한지 확인해주세요.

# 1. 가입일 기준 3일 이내 데이터 필터링을 위한 전처리 및 병합
# 날짜 컬럼들을 datetime 객체로 변환 (연산 가능하도록)
users['signup_date'] = pd.to_datetime(users['signup_date'])
event_logs['event_timestamp'] = pd.to_datetime(event_logs['event_timestamp'])

# 유저 정보와 로그 결합
logs_with_signup = pd.merge(
    event_logs, 
    users[['user_id', 'signup_date']], 
    on='user_id', 
    how='left'
)

# [분석 1] 가입 후 3일 이내, 3개 이상의 레슨 완료
# ---------------------------------------------------------
within_3days = logs_with_signup[
    (logs_with_signup['event_timestamp'] <= logs_with_signup['signup_date'] + timedelta(days=3)) &
    (logs_with_signup['event_name'] == 'lesson_completed')
].copy()

lesson_counts = within_3days.groupby('user_id').size()
# 3개 이상 완료한 유저 ID 추출
power_users = lesson_counts[lesson_counts >= 3].index
users['is_lesson_power_user'] = users['user_id'].isin(power_users)


# [분석 2] 첫 번째 퀴즈 합격 (80점 이상)
# ---------------------------------------------------------
def check_quiz_pass(prop_str):
    try:
        # 문자열인 경우 JSON 파싱, 이미 딕셔너리면 바로 사용
        props = json.loads(prop_str) if isinstance(prop_str, str) else prop_str
        return props.get('score', 0) >= 80
    except:
        return False

quiz_logs = event_logs[event_logs['event_name'] == 'quiz_submitted'].copy()
quiz_logs['is_pass'] = quiz_logs['event_properties'].apply(check_quiz_pass)
passed_users = quiz_logs[quiz_logs['is_pass'] == True]['user_id'].unique()
users['has_passed_quiz'] = users['user_id'].isin(passed_users)


# [분석 3] 온보딩 완료 후 찜(Wishlist) 2개 이상
# ---------------------------------------------------------
wishlist_counts = event_logs[event_logs['event_name'] == 'course_wishlisted'].groupby('user_id').size()
active_wishers = wishlist_counts[wishlist_counts >= 2].index
users['is_active_wisher'] = users['user_id'].isin(active_wishers)


# [결과 통합 및 리프트 계산]
# ---------------------------------------------------------
# 결제 여부 (current_plan이 있는 경우)
users['is_converted'] = users['current_plan'].notnull()

candidates_v2 = ['is_lesson_power_user', 'has_passed_quiz', 'is_active_wisher']
results_v2 = []

for cand in candidates_v2:
    # 각 조건별 평균 결제율 계산
    cvr = users.groupby(cand)['is_converted'].mean() * 100
    
    # 데이터가 부족하여 True/False 중 하나가 없을 경우를 대비한 예외 처리
    f_cvr = cvr[False] if False in cvr.index else 0.1 # 분모 0 방지
    t_cvr = cvr[True] if True in cvr.index else 0
    
    results_v2.append({
        'Candidate': cand,
        'False_CVR': f_cvr,
        'True_CVR': t_cvr,
        'Lift': t_cvr / f_cvr
    })

df_res_v2 = pd.DataFrame(results_v2)
print("--- 1~3번 지표 분석 결과 ---")
print(df_res_v2)

--- 1~3번 지표 분석 결과 ---
              Candidate  False_CVR   True_CVR      Lift
0  is_lesson_power_user  28.648208  39.682540  1.385167
1       has_passed_quiz  21.136622  68.085106  3.221192
2      is_active_wisher  21.297076  81.836903  3.842636


In [9]:
import pandas as pd
import numpy as np

def analyze_ab_test_final(experiment_id, metric_column):
    # 1. 데이터 병합 및 전처리
    exp_data = ab_assignment[ab_assignment['experiment_id'] == experiment_id].copy()
    
    # [핵심] 그룹명 앞뒤의 보이지 않는 공백 제거 및 소문자화
    exp_data['variant'] = exp_data['variant'].str.strip().str.lower()
    
    merged = pd.merge(exp_data, users, on='user_id', how='inner')
    
    # 2. 그룹별 집계
    summary = merged.groupby('variant').agg(
        유저수=('user_id', 'count'),
        전환수=(metric_column, 'sum')
    )
    summary['전환율(CVR)'] = summary['전환수'] / summary['유저수']
    
    # 3. 대조군(Control)과 실험군(Treatment) 추출
    try:
        ctrl = summary.loc['control']
        treat = summary.loc['treatment']
    except KeyError as e:
        print(f"❌ 에러 발생: {e} 그룹을 찾을 수 없습니다. 현재 그룹명: {summary.index.tolist()}")
        return

    # 4. Z-점수 계산 (통계적 유의성 검정)
    p_pool = (ctrl['전환수'] + treat['전환수']) / (ctrl['유저수'] + treat['유저수'])
    se = np.sqrt(p_pool * (1 - p_pool) * (1/ctrl['유저수'] + 1/treat['유저수']))
    
    z_score = (treat['전환율(CVR)'] - ctrl['전환율(CVR)']) / se
    lift = (treat['전환율(CVR)'] / ctrl['전환율(CVR)'] - 1) * 100
    
    print(f"\n📢 [{experiment_id}] 실험 최종 분석 보고서")
    print("-" * 40)
    print(summary)
    print("-" * 40)
    print(f"▶ 리프트(Lift): {lift:.2f}%")
    print(f"▶ Z-Score: {z_score:.4f}")
    
    if abs(z_score) > 1.96:
        status = "🎉 성공! 통계적으로 유의미한 차이가 있습니다." if z_score > 0 else "📉 실패.. 대조군보다 성과가 낮습니다."
        print(f"결과: {status}")
    else:
        print("결과: ⚪ 유의미한 차이가 없습니다. (우연일 가능성이 높음)")

# --- 분석 실행 ---
print("1. 온보딩 리디자인 실험 (EXP-001)")
analyze_ab_test_final('EXP-001', 'is_activated')

print("\n2. 가격 체계 실험 (EXP-002)")
# 가격 실험의 경우 실제 결제 여부('current_plan'이 null이 아닌 유저)를 기준으로 분석합니다.
users['is_paid'] = users['current_plan'].notnull().astype(int)
analyze_ab_test_final('EXP-002', 'is_paid')

분석 모수: 리테인 유저 7120명 / 이탈 유저 9122명

--- 리테인 vs 이탈 유저 행동 비율 비교 (특정 강의 기준 업데이트) ---
               행동 지표  리테인 유저(%)  이탈 유저(%)  차이 (%p)
             1. 회원가입      100.0     100.0      0.0
           2. 온보딩 완료       52.1      39.0     13.1
      3. 레슨 1개 수강 완료       97.9      50.6     47.4
    4. 카테고리 3개 이상 검색       58.8       5.8     52.9
     5. 강의 3개 이상 찜하기       14.4       0.2     14.3
6. 특정 핵심 강의 3개 이상 수강       52.2      11.2     41.1


In [18]:
import pandas as pd
import numpy as np

def run_ab_test_final(exp_id, metric_col):
    # 1. 데이터 필터링 및 병합
    # 실제 데이터에 있는 ID인 'exp_onboarding_v2' 등을 사용합니다.
    target_exp = ab_assignment[ab_assignment['experiment_id'] == exp_id].copy()
    
    # 공백 제거 및 소문자화 (안전장치)
    target_exp['variant'] = target_exp['variant'].str.strip().str.lower()
    
    # users 테이블과 결합
    merged = pd.merge(target_exp, users, on='user_id', how='inner')
    
    # 2. 그룹별 통계 계산
    summary = merged.groupby('variant').agg(
        n=('user_id', 'count'),
        success=(metric_col, 'sum')
    )
    summary['cvr'] = summary['success'] / summary['n']
    
    # 3. Z-검정 수행 (95% 신뢰수준)
    try:
        ctrl = summary.loc['control']
        treat = summary.loc['treatment']
        
        # 합산 비율(Pooled Proportion)
        p_pool = (ctrl['success'] + treat['success']) / (ctrl['n'] + treat['n'])
        # 표준 오차(Standard Error)
        se = np.sqrt(p_pool * (1 - p_pool) * (1/ctrl['n'] + 1/treat['n']))
        
        # Z-점수 공식:
        # $$Z = \frac{\hat{p}_t - \hat{p}_c}{\sqrt{\hat{p}(1-\hat{p})(\frac{1}{n_t} + \frac{1}{n_c})}}$$
        z_score = (treat['cvr'] - ctrl['cvr']) / se
        lift = (treat['cvr'] / ctrl['cvr'] - 1) * 100
        
        print(f"\n📊 [{exp_id}] 실험 분석 결과")
        print("-" * 50)
        print(summary[['n', 'success', 'cvr']])
        print("-" * 50)
        print(f"▶ 리프트(Lift): {lift:.2f}%")
        print(f"▶ Z-Score: {z_score:.4f}")
        
        if abs(z_score) > 1.96:
            print(f"✅ 결과: 통계적으로 유의미한 차이가 발견되었습니다! (Win: {z_score > 0})")
        else:
            print("⚪ 결과: 통계적으로 유의미한 차이가 없습니다. (우연일 가능성 존재)")
            
    except KeyError:
        print(f"❌ '{exp_id}' 실험에서 control/treatment 그룹을 찾을 수 없습니다.")
        print(f"현재 발견된 그룹: {summary.index.tolist()}")

# --- [최종 실행] ---

# 1. 온보딩 리디자인 실험 분석 (Aha Moment 도달률 기준)
run_ab_test_final('exp_onboarding_v2', 'is_activated')

# 2. 가격 체계 실험 분석 (결제 전환율 기준)
run_ab_test_final('exp_pricing_page', 'is_paid')


📊 [exp_onboarding_v2] 실험 분석 결과
--------------------------------------------------
              n  success       cvr
variant                           
control    1182      169  0.142978
treatment  1195      175  0.146444
--------------------------------------------------
▶ 리프트(Lift): 2.42%
▶ Z-Score: 0.2401
⚪ 결과: 통계적으로 유의미한 차이가 없습니다. (우연일 가능성 존재)

📊 [exp_pricing_page] 실험 분석 결과
--------------------------------------------------
             n  success       cvr
variant                          
control    965      269  0.278756
treatment  965      289  0.299482
--------------------------------------------------
▶ 리프트(Lift): 7.43%
▶ Z-Score: 1.0042
⚪ 결과: 통계적으로 유의미한 차이가 없습니다. (우연일 가능성 존재)
